# Syntatic and Semantic Text Analysis

**Grammar Parsing**:  
untuk analisis struktur kalimat (kyk pos tagging), ngebagi kalimat ke pecahan yang lebih kecil dgn rules tertentu  
  
*Example:*  
*"The dog saw a man in the park"*  
  
S  
NP + VP  
Det + N | V + NP + PP  
"The" | "Dog" | "Saw" | Det + Noun | P + NP  
"The" | "Dog" | "Saw" | "a" | "man" | "in" | Det + N  
"The" | "Dog" | "Saw" | "a" | "man" | "in" | "the" | "park"

**Dependency Parsing**  
menentukan hubungan antara suatu kata dengan kata lain dalam kalimat

**NER (Name Entity Recognition)**  
untuk ekstrasi bagian yang penting dari kalimat

### Part 1 (Grammar Parsing)

In [16]:
from nltk import CFG  # context free grammar
from nltk.parse import ChartParser
from nltk.tokenize import word_tokenize

In [17]:
sentences = [
    "the snail slide the snake",
    "i saw a stick",
    "the snake slid the stick in the sleigh",
    "i spoil the snail with the stick",
    "i saw a man with the telescope",
]

In [23]:
grammar = CFG.fromstring('''
    S -> NP VP
    NP -> Det N | 'i' | Det N PP
    VP -> V NP | V NP PP  
    PP -> P NP
    Det -> 'the' | 'a'
    N -> 'snail' | 'snake' | 'stick' | 'sleigh' | 'man' | 'telescope'
    V -> 'slide' | 'saw' | 'slid' | 'spoil'
    P -> 'in' | 'with'
''')

In [24]:
def extract_information(parse_tree):
    for subtree in parse_tree.subtrees():
        if subtree.label() == 'NP':
            print(f"Found Noun Phrases: {' '.join(subtree.leaves())}")

In [31]:
def demonstrate_parsing(sentence, grammar):
    words = word_tokenize(sentence)
    parser = ChartParser(grammar)
    
    try:
        parses = list(parser.parse(words))  # biar semuanya bisa ketulis di dalem list, ngga cmn 1
        if parses:
            for i, tree in enumerate(parser.parse(words)):
                print(tree, '\n')
                tree.pretty_print()
                
                print('Extracted information')
                extract_information(tree[i])
        else:
            print('No parses found')
    except:
        print('Error during parsing')

In [32]:
for i, sentence in enumerate(sentences):
    print(f'Sentence {i+1}: {sentence}')
    demonstrate_parsing(sentence, grammar)
    # print('\n')

Sentence 1: the snail slide the snake
(S (NP (Det the) (N snail)) (VP (V slide) (NP (Det the) (N snake)))) 

                S                
      __________|____             
     |               VP          
     |           ____|___         
     NP         |        NP      
  ___|____      |     ___|____    
Det       N     V   Det       N  
 |        |     |    |        |   
the     snail slide the     snake

Extracted information
Found Noun Phrases: the snail
Sentence 2: i saw a stick
(S (NP i) (VP (V saw) (NP (Det a) (N stick)))) 

         S               
  _______|___             
 |           VP          
 |    _______|___         
 |   |           NP      
 |   |        ___|____    
 NP  V      Det       N  
 |   |       |        |   
 i  saw      a      stick

Extracted information
Found Noun Phrases: i
Sentence 3: the snake slid the stick in the sleigh
(S
  (NP (Det the) (N snake))
  (VP
    (V slid)
    (NP (Det the) (N stick))
    (PP (P in) (NP (Det the) (N sleigh)))

### Part 2 (Dependency Parsing)

In [34]:
import spacy

def process(text):
    nlp = spacy.load('en_core_web_sm')
    doc = nlp(text)
    return doc

In [36]:
def parse_dependency_tree(sentence):
    print("Dependency Parsing Tree")
    doc = process(sentence)
    
    for token in doc:
        print(f"{token.text} --{token.dep_}--> {token.head.text} ({token.pos_})")
        #     {word} --{dependency of word}--> {head word} {part of speech tag}

In [37]:
sentence = "Elon Musk founded Tesla, and the headquarters are in Texas"

In [38]:
parse_dependency_tree(sentence)

Dependency Parsing Tree
Elon --compound--> Musk (PROPN)
Musk --nsubj--> founded (PROPN)
founded --ROOT--> founded (VERB)
Tesla --dobj--> founded (PROPN)
, --punct--> founded (PUNCT)
and --cc--> founded (CCONJ)
the --det--> headquarters (DET)
headquarters --nsubj--> are (NOUN)
are --conj--> founded (AUX)
in --prep--> are (ADP)
Texas --pobj--> in (PROPN)


### Part 3 (NER with SpaCy) -> materi UAP

In [39]:
def extract_ner(sentence):
    doc = process(sentence)
    
    named_entities = {
        'persons': [ent.text for ent in doc.ents if ent.label_ == "PERSON"],
        'gpes': [ent.text for ent in doc.ents if ent.label_ == "GPE"],
        'organizations': [ent.text for ent in doc.ents if ent.label_ == "ORG"],
    }
    
    return named_entities

In [42]:
print("Named Entities")

named_entities = extract_ner(sentence)
print("Person", named_entities['persons'])
print("GPE", named_entities['gpes'])
print("Organizations", named_entities['organizations'])

Named Entities
Person ['Elon Musk']
GPE ['Texas']
Organizations ['Tesla']


#### All NER

In [43]:
paragraph = '''
Sea monsters are the stuff of legend - lurking not just in the depths of the oceans, but also the darker corners of our minds. What is it that draws us to these creatures?

"This inhuman place makes human monsters," wrote Stephen King in his novel The Shining. Many academics agree that monsters lurk in the deepest recesses, they prowl through our ancestral minds appearing in the half-light, under the bed - or at the bottom of the sea.

"They don't really exist, but they play a huge role in our mindscapes, in our dreams, stories, nightmares, myths and so on," says Matthias Classen, assistant professor of literature and media at Aarhus University in Denmark, who studies monsters in literature. "Monsters say something about human psychology, not the world."

One Norse legend talks of the Kraken, a deep sea creature that was the curse of fishermen. If sailors found a place with many fish, most likely it was the monster that was driving them to the surface. If it saw the ship it would pluck the hapless sailors from the boat and drag them to a watery grave.
This terrifying legend occupied the mind and pen of the poet Alfred Lord Tennyson too. In his short 1830 poem The Kraken he wrote: "Below the thunders of the upper deep, / Far far beneath in the abysmal sea, / His ancient, dreamless, uninvaded sleep / The Kraken sleepeth."

The deeper we travel into the ocean, the deeper we delve into our own psyche. And when we can go no further - there lurks the Kraken.

Most likely the Kraken is based on a real creature - the giant squid. The huge mollusc takes pride of place as the personification of the terrors of the deep sea. Sailors would have encountered it at the surface, dying, and probably thrashing about. It would have made a weird sight, "about the most alien thing you can imagine," says Edith Widder, CEO at the Ocean Research and Conservation Association.

"It has eight lashing arms and two slashing tentacles growing straight out of its head and it's got serrated suckers that can latch on to the slimiest of prey and it's got a parrot beak that can rip flesh. It's got an eye the size of your head, it's got a jet propulsion system and three hearts that pump blue blood."

The giant squid continued to dominate stories of sea monsters with the famous 1870 novel, Twenty Thousand Leagues Under the Sea, by Jules Verne. Verne's submarine fantasy is a classic story of puny man against a gigantic squid.

The monster needed no embellishment - this creature was scary enough, and Verne incorporated as much fact as possible into the story, says Emily Alder from Edinburgh Napier University. "Twenty Thousand Leagues Under the Sea and another contemporaneous book, Victor Hugo's Toilers of the Sea, both tried to represent the giant squid as they might have been actual zoological animals, much more taking the squid as a biological creature than a mythical creature." It was a given that the squid was vicious and would readily attack humans given the chance.

That myth wasn't busted until 2012, when Edith Widder and her colleagues were the first people to successfully film giant squid under water and see first-hand the true character of the monster of the deep. They realised previous attempts to film squid had failed because the bright lights and noisy thrusters on submersibles had frightened them away.

By quietening down the engines and using bioluminescence to attract it, they managed to see this most extraordinary animal in its natural habitat. It serenely glided into view, its body rippled with metallic colours of bronze and silver. Its huge, intelligent eye watched the submarine warily as it delicately picked at the bait with its beak. It was balletic and mesmeric. It could not have been further from the gnashing, human-destroying creature of myth and literature. In reality this is a gentle giant that is easily scared and pecks at its food.

Another giant squid lies peacefully in the Natural History Museum in London, in the Spirit Room, where it is preserved in a huge glass case. In 2004 it was caught in a fishing net off the Falkland Islands and died at the surface. The crew immediately froze its body and it was sent to be preserved in the museum by the Curator of Molluscs, Jon Ablett. It is called Archie, an affectionate short version of its Latin name Architeuthis dux. It is the longest preserved specimen of a giant squid in the world.

"It really has brought science to life for many people," says Ablett. "Sometimes I feel a bit overshadowed by Archie, most of my work is on slugs and snails but unfortunately most people don't want to talk about that!"

And so today we can watch Archie's graceful relative on film and stare Archie herself (she is a female) eye-to-eye in a museum. But have we finally slain the monster of the deep? Now we know there is nothing to be afraid of, can the Kraken finally be laid to rest? Probably not says Classen. "We humans are afraid of the strangest things. They don't need to be realistic. There's no indication that enlightenment and scientific progress has banished the monsters from the shadows of our imaginations. We will continue to be afraid of very strange things, including probably sea monsters."

Indeed we are. The Kraken made a fearsome appearance in the blockbuster series Pirates of the Caribbean. It forced Captain Jack Sparrow to face his demons in a terrifying face-to-face encounter. Pirates needed the monstrous Kraken, nothing else would do. Or, as the German film director Werner Herzog put it, "What would an ocean be without a monster lurking in the dark? It would be like sleep without dreams."
'''

In [45]:
doc = process(paragraph)

categories = {}

for ent in doc.ents:
    label = ent.label_
    if label not in categories:
        categories[label] = []
    categories[label].append(ent.text)

for label, word in categories.items():
    print(f"{label}: {', '.join(word)}")

LOC: Sea
PERSON: Stephen King, Matthias Classen, Alfred Lord Tennyson, Edith Widder, Jules Verne, Verne, Verne, Emily Alder, Edith Widder, Jon Ablett, Ablett, Classen, Jack Sparrow, Werner Herzog
WORK_OF_ART: The Shining, Victor Hugo's Toilers of the Sea, the Curator of Molluscs, Pirates of the Caribbean
CARDINAL: half, One, eight, two, three, Twenty Thousand
ORG: Aarhus University, Norse, Kraken, Kraken, Kraken, Kraken, the Ocean Research and Conservation Association, Edinburgh Napier University, the Natural History Museum, Archie, Archie, Kraken, Kraken, Kraken
GPE: Denmark, London, the Falkland Islands
DATE: 1830, 1870, 2012, 2004, today
MONEY: Twenty Thousand Leagues
ORDINAL: first, first
PRODUCT: Archie, Archie
LANGUAGE: Latin
NORP: German
